# Конкурентная запись в DuckLake

DuckLake обеспечивает ACID-транзакции через PostgreSQL-каталог.
Несколько процессов могут писать одновременно — каждый получает
свой снэпшот. Конфликты разрешаются на уровне каталога.

В production-режиме это два Airflow-воркера, пишущих бронирования параллельно.

In [ ]:
import os, sys, threading, time
sys.path.insert(0, '..')

os.environ.setdefault('DUCKLAKE_PG_HOST', 'localhost')
os.environ.setdefault('DUCKLAKE_PG_DB', 'ducklake_catalog')
os.environ.setdefault('DUCKLAKE_PG_USER', 'ducklake')
os.environ.setdefault('DUCKLAKE_PG_PASSWORD', 'ducklake_secret_change_me')
os.environ.setdefault('MINIO_ACCESS_KEY', 'minioadmin')
os.environ.setdefault('MINIO_SECRET_KEY', 'minioadmin')
os.environ.setdefault('MINIO_ENDPOINT', 'http://localhost:9000')
os.environ.setdefault('MINIO_BUCKET', 'ducklake-flights')

from src.generators.connection import get_ducklake_connection
import pandas as pd
print('Ready.')

## Количество бронирований до теста

In [ ]:
conn_check = get_ducklake_connection(read_only=True)
count_before = conn_check.execute("SELECT COUNT(*) FROM flights.bookings").fetchone()[0]
conn_check.close()
print(f'Бронирований до теста: {count_before:,}')

## Параллельная запись из двух потоков

In [ ]:
from datetime import datetime, timezone
from src.generators.booking_generator import (
    _insert_bookings, _insert_passengers, _insert_price_history,
    generate_bookings_batch,
)
from src.generators.flight_generator import _load_routes_and_airports

results = {}
errors = {}

def worker(worker_id: int, batch_size: int = 30):
    """Имитация Airflow-воркера: открывает соединение, пишет батч бронирований."""
    try:
        conn = get_ducklake_connection()

        # Загружаем рейсы
        rows = conn.execute("""
            SELECT flight_id, flight_number, airline_iata,
                   src_airport_iata, dst_airport_iata,
                   scheduled_departure, scheduled_arrival,
                   status, aircraft_type, total_seats, flight_date
            FROM flights.flights
            WHERE status NOT IN ('cancelled', 'arrived')
            LIMIT 50
        """).fetchall()

        cols = ['flight_id','flight_number','airline_iata','src_airport_iata','dst_airport_iata',
                'scheduled_departure','scheduled_arrival','status','aircraft_type','total_seats','flight_date']
        flights = [dict(zip(cols, r)) for r in rows]

        airports_rows = conn.execute(
            "SELECT iata_code, latitude, longitude FROM flights.airports"
        ).fetchall()
        airports = {r[0]: {'iata_code': r[0], 'latitude': r[1], 'longitude': r[2]} for r in airports_rows if r[0]}

        bookings, passengers, prices = generate_bookings_batch(
            flights, airports, target_count=batch_size,
            batch_time=datetime.now(timezone.utc)
        )

        _insert_passengers(conn, passengers)
        _insert_bookings(conn, bookings)
        _insert_price_history(conn, prices)
        conn.close()

        results[worker_id] = len(bookings)
        print(f'Worker {worker_id}: записал {len(bookings)} бронирований')

    except Exception as e:
        errors[worker_id] = str(e)
        print(f'Worker {worker_id}: ОШИБКА — {e}')


# Запускаем двух воркеров одновременно
t1 = threading.Thread(target=worker, args=(1,))
t2 = threading.Thread(target=worker, args=(2,))

start = time.time()
t1.start()
t2.start()
t1.join()
t2.join()
elapsed = time.time() - start

print(f'\nВремя: {elapsed:.1f}с')
print(f'Ошибок: {len(errors)}')
if errors:
    for wid, err in errors.items():
        print(f'  Worker {wid}: {err}')

## Проверка: все записи попали в DuckLake

In [ ]:
conn_check = get_ducklake_connection(read_only=True)
count_after = conn_check.execute("SELECT COUNT(*) FROM flights.bookings").fetchone()[0]
conn_check.close()

total_written = sum(results.values())
actual_delta = count_after - count_before

print(f'Бронирований до:      {count_before:,}')
print(f'Бронирований после:   {count_after:,}')
print(f'Записано воркерами:   {total_written}')
print(f'Фактический прирост:  {actual_delta}')
print()
if actual_delta >= total_written * 0.9:
    print('✓ Конкурентная запись успешна. Данные консистентны.')
else:
    print('⚠ Часть записей потеряна или дублирована — проверьте логи.')

## Снэпшоты двух параллельных транзакций

In [ ]:
conn_check = get_ducklake_connection(read_only=True)
snaps = conn_check.execute("""
    SELECT snapshot_id, snapshot_time
    FROM ducklake_snapshots('flights')
    ORDER BY snapshot_id DESC
    LIMIT 5
""").df()
conn_check.close()

print('Последние снэпшоты (два новых — от двух воркеров):')
display(snaps)

In [ ]:
print('Done.')